# **Evaluasi Tengah Semester Genap 2024-2025 Praktikum Machine Vision**
Disusun oleh:
| Nama: | Dhoifulloh Ammar Rozaan, Moch. Reihan Yumna Wahyudi, Faizal Dewayana putra |
| - | - |
| NRP: | 0922040029, 0922040043, 0922040047 |
| Kelas: | TOVI-B |
| Mata kuliah: | Praktikum Machine Vision |
| Hari/Tanggal: | Rabu/23 April 2025 |
| Pekan ke: | ETS|
| Dosen Pengampu: | Agus Khumaidi, S.ST., M.T. & Mustika Kurnia Mayangsari, S.T., M.Tr.T. |

---

In [ ]:
import cv2
import tkinter as tk
from tkinter import filedialog
from PIL import Image, ImageTk
import easyocr
import numpy as np

# Inisialisasi OCR
reader = easyocr.Reader(['en'])

# Fungsi deteksi plat dari gambar
def detect_plate_from_image():
    file_path = filedialog.askopenfilename(
        filetypes=[("Image files", "*.jpg;*.jpeg;*.png")])
    if not file_path:
        return

    image = cv2.imread(file_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Deteksi tepi dan kontur
    edges = cv2.Canny(gray, 100, 200)
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    plate_text = "Plat tidak terdeteksi"
    best_result = None
    x, y = 0, 0  # Default koordinat

    possible_plates = []

    for cnt in contours:
        approx = cv2.approxPolyDP(cnt, 0.02 * cv2.arcLength(cnt, True), True)
        if len(approx) == 4:
            x, y, w, h = cv2.boundingRect(cnt)
            aspect_ratio = w / float(h)
            if 2 < aspect_ratio < 6 and w > 100:
                possible_plates.append((x, y, w, h))

    if possible_plates:
        # Ambil kontur terbesar (kemungkinan besar plat)
        x, y, w, h = sorted(possible_plates, key=lambda b: b[2] * b[3], reverse=True)[0]
        plate_img = image[y:y + h, x:x + w]

        # Preprocessing sebelum OCR
        plate_gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)
        plate_blur = cv2.GaussianBlur(plate_gray, (5, 5), 0)
        _, plate_thresh = cv2.threshold(plate_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # OCR
        results = reader.readtext(plate_thresh)

        # Filter hasil dengan confidence > 0.5
        if results:
            filtered = [text for (_, text, conf) in results if conf > 0.5]
            plate_text = " ".join(filtered) if filtered else "Plat tidak terbaca (akurasi rendah)"

            # Cetak hasil ke terminal
            for (bbox, text, confidence) in results:
                print(f"Teks: {text}, Akurasi: {confidence:.2f}")

        # Kotak hijau di sekitar plat
        cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Tampilkan teks pada gambar
        cv2.putText(image, plate_text, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Tampilkan di GUI (resize agar tidak zoom berlebihan)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(image_rgb)
    img_pil.thumbnail((800, 400))  # Resize gambar secara proporsional
    img_tk = ImageTk.PhotoImage(img_pil)

    label_image.configure(image=img_tk)
    label_image.image = img_tk
    label_result.configure(text=f"Hasil Deteksi: {plate_text}")

# GUI setup
root = tk.Tk()
root.title("Deteksi Plat Nomor dari Foto HP")

btn_load = tk.Button(root, text="Pilih Foto Kendaraan", command=detect_plate_from_image)
btn_load.pack(pady=10)

label_image = tk.Label(root)
label_image.pack()

label_result = tk.Label(root, text="Hasil Deteksi: -", font=("Arial", 14))
label_result.pack(pady=10)

root.mainloop()